### Ajouter prix au m2 par maison

In [1]:
import os
import pandas as pd

data_dir = os.path.join('..', 'csv', 'STEP02')

# Process each CSV file in STEP02
for file in os.listdir(data_dir):
    if file.endswith('.csv'):
        path = os.path.join(data_dir, file)
        df = pd.read_csv(path)
        
        # Nettoyage numérique
        df['Taille'] = pd.to_numeric(df['Taille'].astype(str).str.replace(' ', ''), errors='coerce')
        df['Prix'] = pd.to_numeric(df['Prix'].astype(str).str.replace(' ', ''), errors='coerce')
        
        # Assuming columns 'Prix' (price) and 'Taille' (area in m²)
        # Calculate 'Prix au m2'
        df['Prix au m2'] = df['Prix'] / df['Taille']
        
        # Arrondir à l'entier
        df['Prix au m2'] = df['Prix au m2'].round(0)
        
        # Save the updated DataFrame back to the same file
        df.to_csv(path, index=False)

### Vérifier si d'autres villes ne se sont pas glissées dans le scraping

In [7]:
import re
import os
import pandas as pd

# Utiliser le répertoire STEP02 défini précédemment
data_dir = os.path.join('..', 'csv', 'STEP02')

# Liste de tous les départements traités
depts_traitees = []

# Dictionnaire pour stocker les erreurs par département
erreurs_par_dept = {}

# Process each CSV file in STEP02
for file in os.listdir(data_dir):
    if file.endswith('.csv'):
        dept = file.split('dept')[1].split('.csv')[0]
        depts_traitees.append(dept)
        path = os.path.join(data_dir, file)
        df = pd.read_csv(path)
        
        erreurs = []
        
        for idx, row in df.iterrows():
            lieu = str(row['Lieu'])
            # Extraire le code postal à la fin (supposé être des chiffres après un espace)
            match = re.search(r'\b(\d{5})\b$', lieu)
            if match:
                postal = match.group(1)
                if postal[:2] != dept:
                    erreurs.append({
                        'Index': idx,
                        'Lieu': lieu,
                        'Code postal': postal,
                        'Dépt attendu': dept
                    })
        
        erreurs_par_dept[dept] = erreurs

# Créer le répertoire STEP03
output_dir = os.path.join('..', 'csv', 'STEP03')
os.makedirs(output_dir, exist_ok=True)

# Créer les CSV STEP03 en supprimant les erreurs
for dept in depts_traitees:
    input_path = os.path.join(data_dir, f'STEP02_maisons_dept{dept}.csv')
    output_path = os.path.join(output_dir, f'STEP03_maisons_dept{dept}.csv')
    
    df = pd.read_csv(input_path)
    errs = erreurs_par_dept[dept]
    if errs:
        indices_to_drop = [err['Index'] for err in errs]
        df_clean = df.drop(indices_to_drop)
        nb_supprimees = len(indices_to_drop)
    else:
        df_clean = df
        nb_supprimees = 0
    
    df_clean.to_csv(output_path, index=False)
    print(f"Département {dept} : {nb_supprimees} erreurs supprimées. CSV nettoyé sauvegardé dans {output_path}")

# Afficher les erreurs par département
for dept in sorted(depts_traitees):
    errs = erreurs_par_dept[dept]
    print(f"\nDépartement {dept} : {len(errs)} erreurs trouvées")
    if errs:
        for err in errs[:5]:  # Afficher seulement les 5 premiers exemples
            print(f"  Index {err['Index']}: {err['Lieu']} (Code: {err['Code postal']}, Attendu: {err['Dépt attendu']})")
        if len(errs) > 5:
            print(f"  ... et {len(errs) - 5} autres erreurs")

Département 22 : 0 erreurs supprimées. CSV nettoyé sauvegardé dans ..\csv\STEP03\STEP03_maisons_dept22.csv
Département 29 : 0 erreurs supprimées. CSV nettoyé sauvegardé dans ..\csv\STEP03\STEP03_maisons_dept29.csv
Département 35 : 594 erreurs supprimées. CSV nettoyé sauvegardé dans ..\csv\STEP03\STEP03_maisons_dept35.csv
Département 56 : 0 erreurs supprimées. CSV nettoyé sauvegardé dans ..\csv\STEP03\STEP03_maisons_dept56.csv

Département 22 : 0 erreurs trouvées

Département 29 : 0 erreurs trouvées

Département 35 : 594 erreurs trouvées
  Index 4777: Flize 08160 (Code: 08160, Attendu: 35)
  Index 4778: Ledignan 30350 (Code: 30350, Attendu: 35)
  Index 4779: Lisieux 14100 (Code: 14100, Attendu: 35)
  Index 4780: Lisieux 14100 (Code: 14100, Attendu: 35)
  Index 4781: Lisieux 14100 (Code: 14100, Attendu: 35)
  ... et 589 autres erreurs

Département 56 : 0 erreurs trouvées


### Supprimer maisons avec "ferme" "rénover" "renover"

In [8]:
import re
import pandas as pd
import os

# Charger tous les CSV STEP03 pour nettoyage en place
files = [
    '../csv/STEP03/STEP03_maisons_dept22.csv',
    '../csv/STEP03/STEP03_maisons_dept29.csv',
    '../csv/STEP03/STEP03_maisons_dept35.csv',
    '../csv/STEP03/STEP03_maisons_dept56.csv'
]

# Charger les DataFrames
dfs = [pd.read_csv(f, encoding='utf-8') for f in files]

# Fonction de filtrage
def filtrer_maisons(df):
    # On retire les lignes dont le titre contient "ferme", "rénover" ou "renover" (insensible à la casse)
    masque = ~df['Nom'].str.contains(r'ferme|rénover|renover', flags=re.IGNORECASE, na=False)
    return df[masque]

# Appliquer le filtre à chaque DataFrame de dfs
dfs_filtrés = [filtrer_maisons(d) for d in dfs]

# Sauvegarder dans les fichiers STEP03 (en place) et afficher le nombre de maisons supprimées
for i, df_filtré in enumerate(dfs_filtrés):
    dept_num = files[i].split('dept')[1].split('.csv')[0]
    nombre_avant = len(dfs[i])
    nombre_apres = len(df_filtré)
    nombre_supprimees = nombre_avant - nombre_apres
    output_path = files[i]  # Même fichier
    df_filtré.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Département {dept_num}: {nombre_supprimees} maisons supprimées ({nombre_avant} → {nombre_apres})")
    print(f"Fichier mis à jour : {output_path}")



Département 22: 87 maisons supprimées (4225 → 4138)
Fichier mis à jour : ../csv/STEP03/STEP03_maisons_dept22.csv
Département 29: 121 maisons supprimées (5335 → 5214)
Fichier mis à jour : ../csv/STEP03/STEP03_maisons_dept29.csv
Département 35: 121 maisons supprimées (5273 → 5152)
Fichier mis à jour : ../csv/STEP03/STEP03_maisons_dept35.csv
Département 56: 147 maisons supprimées (5898 → 5751)
Fichier mis à jour : ../csv/STEP03/STEP03_maisons_dept56.csv


### Supprimer les maisons doublons

In [9]:
import re
import pandas as pd

# Charger tous les CSV STEP03
files = [
    '../csv/STEP03/STEP03_maisons_dept22.csv',
    '../csv/STEP03/STEP03_maisons_dept29.csv',
    '../csv/STEP03/STEP03_maisons_dept35.csv',
    '../csv/STEP03/STEP03_maisons_dept56.csv'
]

# Charger les DataFrames
dfs = [pd.read_csv(f) for f in files]

# Colonnes à vérifier pour les doublons (toutes sauf 'Nom' et 'Lien')
colonnes_a_verifier = ['Code INSEE', 'Page Lien', 'Prix', 'Lieu', 'Taille', 'Taille_terrain', 'Pieces', 'Prix au m2']

total_supprimees = 0

for idx, df in enumerate(dfs):
    nombre_avant = len(df)

    # Afficher les maisons dont seules les colonnes 'Nom' et 'Lien' diffèrent (potentiels doublons)
    doublons = df[df.duplicated(subset=colonnes_a_verifier, keep=False)]
    if not doublons.empty:
        print(f"\nDépartement {files[idx].split('dept')[1].split('.csv')[0]} : {len(doublons)} doublons potentiels trouvés")
        # Afficher les groupes de doublons avec leurs titres et liens
        grouped = doublons.groupby(colonnes_a_verifier)
        for _, group in grouped:
            if len(group) > 1:
                # Afficher seulement Nom, Prix et Lien avec lien cliquable
                df_affichage = group[['Nom', 'Prix', 'Lien']].copy()
                print(df_affichage.to_string(index=False))
                print("\nLiens:")
                for _, row in df_affichage.iterrows():
                    print(f"- {row['Nom']}: {row['Lien']}")
    else:
        print(f"\nDépartement {files[idx].split('dept')[1].split('.csv')[0]} : Aucun doublon potentiel trouvé")

    # Supprimer les doublons (garder la première occurrence)
    df_sans_doublons = df.drop_duplicates(subset=colonnes_a_verifier, keep='first')
    nombre_apres = len(df_sans_doublons)
    nombre_supprimees = nombre_avant - nombre_apres

    if nombre_supprimees > 0:
        print(f"  → {nombre_supprimees} doublons supprimés ({nombre_avant} → {nombre_apres})")

    # Sauvegarder le fichier nettoyé
    df_sans_doublons.to_csv(files[idx], index=False)
    total_supprimees += nombre_supprimees

print(f"\n=== TOTAL : {total_supprimees} doublons supprimés au total ===")


Département 22 : 117 doublons potentiels trouvés
                  Nom     Prix                                                                              Lien
Maison 120m² à callac 219900.0 https://www.etreproprio.com/immobilier-23429848-vente-maison-120m-a-callac-callac
Maison 120m² à callac 219900.0 https://www.etreproprio.com/immobilier-23165537-vente-maison-120m-a-callac-callac

Liens:
- Maison 120m² à callac: https://www.etreproprio.com/immobilier-23429848-vente-maison-120m-a-callac-callac
- Maison 120m² à callac: https://www.etreproprio.com/immobilier-23165537-vente-maison-120m-a-callac-callac
                 Nom     Prix                                                                            Lien
Maison 138m² à dinan 546000.0 https://www.etreproprio.com/immobilier-23823118-vente-maison-138m-a-dinan-dinan
Maison 138m² à dinan 546000.0 https://www.etreproprio.com/immobilier-23817313-vente-maison-138m-a-dinan-dinan

Liens:
- Maison 138m² à dinan: https://www.etreproprio.com

### Outliers ville précise (pour info)

In [1]:
import pandas as pd

# Configurer pandas pour afficher les colonnes longues en entier
pd.set_option('display.max_colwidth', None)

# Demander à l'utilisateur la ville à chercher
ville = input("Entrez le nom de la ville à chercher: ")

# Charger tous les CSV STEP03
files = [
    '../csv/STEP03/STEP03_maisons_dept22.csv',
    '../csv/STEP03/STEP03_maisons_dept29.csv',
    '../csv/STEP03/STEP03_maisons_dept35.csv',
    '../csv/STEP03/STEP03_maisons_dept56.csv'
]

dfs = [pd.read_csv(f, encoding='utf-8') for f in files]
df = pd.concat(dfs, ignore_index=True)

# Filtrer les maisons pour la ville donnée
maisons_ville = df[df['Lieu'].str.contains(ville, na=False)].copy()

print(f"Maisons à {ville} ({len(maisons_ville)} au total):")
if len(maisons_ville) > 0:
    # Limiter l'affichage pour le tableau principal
    pd.set_option('display.max_colwidth', 50)
    display(maisons_ville.sort_values('Prix au m2'))
    # Remettre à None pour les prints suivants
    pd.set_option('display.max_colwidth', None)

    # Nettoyage pour calculs
    maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')

    # Moyenne avec outliers
    mean_ville = maisons_ville['Prix au m2'].mean()
    print(f"\nMoyenne {ville}: {mean_ville:.2f} €/m²")

    # --- Détection outliers avec IQR ---
    Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
    Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
    IQR_ville = Q3_ville - Q1_ville

    print(f"IQR - Q1: {Q1_ville:.2f}, Q3: {Q3_ville:.2f}, IQR: {IQR_ville:.2f}")

    lower_ville = Q1_ville - 1.5 * IQR_ville
    upper_ville = Q3_ville + 1.5 * IQR_ville

    outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
    print(f"Outliers selon IQR ({len(outliers_ville)}):")
    print(outliers_ville[['Prix au m2', 'Prix', 'Taille', 'Lien']].sort_values('Prix au m2'))

    # Moyenne sans outliers
    df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
    mean_sans_outliers = df_sans_outliers['Prix au m2'].mean()
    print(f"Moyenne sans outliers: {mean_sans_outliers:.2f} €/m²")

else:
    print("Aucune maison trouvée dans le dataset pour cette ville.")

Maisons à Ergue-Gaberic (37 au total):


,Nom,Lien,Code INSEE,Page Lien,Prix,Lieu,Taille,Taille_terrain,Pieces,Prix au m2
5323,Immeuble de rapport sur ergué-gabéric,https://www.etreproprio.com/immobilier-2360682...,29051,https://www.etreproprio.com/annonces/th.lc2905...,291500.0,Ergue-Gaberic 29500,329.0,601,10.0,886.0
5309,Ergue-gaberic maison individuelle sur terrain ...,https://www.etreproprio.com/immobilier-2353244...,29051,https://www.etreproprio.com/annonces/th.lc2905...,199900.0,Ergue-Gaberic 29500,170.0,630,9.0,1176.0
5312,Maison traditionnelle aux portes de quimper!,https://www.etreproprio.com/immobilier-2375721...,29051,https://www.etreproprio.com/annonces/th.lc2905...,176560.0,Ergue-Gaberic 29500,115.0,500,5.0,1535.0
5337,Maison 223m² à ergue-gaberic,https://www.etreproprio.com/immobilier-2310135...,29051,https://www.etreproprio.com/annonces/th.lc2905...,354000.0,Ergue-Gaberic 29500,223.0,1 447,8.0,1587.0
5338,Maison familiale ou emplacement de choix pour ...,https://www.etreproprio.com/immobilier-2178843...,29051,https://www.etreproprio.com/annonces/th.lc2905...,354000.0,Ergue-Gaberic 29500,223.0,1 500,8.0,1587.0
5347,Le rouillen,https://www.etreproprio.com/immobilier-2102068...,29051,https://www.etreproprio.com/annonces/th.lc2905...,174900.0,Ergue-Gaberic 29500,110.0,759,4.0,1590.0
5343,Maison familiale de 6 chambres lestonan - ergu...,https://www.etreproprio.com/immobilier-2287691...,29051,https://www.etreproprio.com/annonces/th.lc2905...,336000.0,Ergue-Gaberic 29500,197.0,1 115,8.0,1706.0
5332,Maison 134m² à ergue-gaberic,https://www.etreproprio.com/immobilier-2255729...,29051,https://www.etreproprio.com/annonces/th.lc2905...,245000.0,Ergue-Gaberic 29500,134.0,850,7.0,1828.0
5311,Maison ergué gaberic- 4 chambres - 100m²,https://www.etreproprio.com/immobilier-2375721...,29051,https://www.etreproprio.com/annonces/th.lc2905...,186375.0,Ergue-Gaberic 29500,100.0,603,5.0,1864.0
5313,Maison familiale 129 m² - ergué-gabéric,https://www.etreproprio.com/immobilier-2375721...,29051,https://www.etreproprio.com/annonces/th.lc2905...,248630.0,Ergue-Gaberic 29500,130.0,728,7.0,1913.0



Moyenne Ergue-Gaberic: 2318.16 €/m²
IQR - Q1: 1913.00, Q3: 2697.00, IQR: 784.00
Outliers selon IQR (0):
Empty DataFrame
Columns: [Prix au m2, Prix, Taille, Lien]
Index: []
Moyenne sans outliers: 2318.16 €/m²


### Outliers département

In [15]:
# Calculer les outliers par ville pour un département donné
import pandas as pd
import os

# Configurer pandas pour limiter l'affichage des colonnes longues
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', None)

# Demander le département
dept = input("Entrez le numéro du département (22, 29, 35 ou 56): ")

# Charger le CSV correspondant
file_path = f'../csv/STEP03/STEP03_maisons_dept{dept}.csv'
df = pd.read_csv(file_path, encoding='utf-8')

# Obtenir les villes uniques
villes = df['Lieu'].unique()

# Liste pour stocker les résultats
resultats = []
# Liste pour stocker les dataframes nettoyés
dfs_clean = []

for ville in villes:
    maisons_ville = df[df['Lieu'] == ville].copy()
    
    if len(maisons_ville) > 0:
        # Nettoyage pour calculs
        maisons_ville['Prix au m2'] = pd.to_numeric(maisons_ville['Prix au m2'], errors='coerce')
        maisons_ville = maisons_ville.dropna(subset=['Prix au m2'])
        
        nombre_maisons = len(maisons_ville)
        moyenne_avant = maisons_ville['Prix au m2'].mean()
        
        if len(maisons_ville) >= 4:  # Au moins 4 points pour calculer IQR
            # --- Détection outliers avec IQR ---
            Q1_ville = maisons_ville['Prix au m2'].quantile(0.25)
            Q3_ville = maisons_ville['Prix au m2'].quantile(0.75)
            IQR_ville = Q3_ville - Q1_ville

            lower_ville = Q1_ville - 1.5 * IQR_ville
            upper_ville = Q3_ville + 1.5 * IQR_ville

            outliers_ville = maisons_ville[(maisons_ville['Prix au m2'] < lower_ville) | (maisons_ville['Prix au m2'] > upper_ville)]
            nombre_outliers = len(outliers_ville)
            
            # Moyenne après outliers
            df_sans_outliers = maisons_ville[~maisons_ville.index.isin(outliers_ville.index)]
            moyenne_apres = df_sans_outliers['Prix au m2'].mean() if len(df_sans_outliers) > 0 else 0
        else:
            nombre_outliers = 0
            moyenne_apres = moyenne_avant  # Pas d'outliers retirés
            df_sans_outliers = maisons_ville  # Garder tout
    else:
        nombre_maisons = 0
        nombre_outliers = 0
        moyenne_avant = 0
        moyenne_apres = 0
        df_sans_outliers = maisons_ville  # Vide
    
    resultats.append({
        'Ville': ville, 
        'Nombre de maisons': nombre_maisons, 
        'Nombre d\'outliers': nombre_outliers,
        'Moyenne avant outliers (€/m²)': round(moyenne_avant, 2),
        'Moyenne après outliers (€/m²)': round(moyenne_apres, 2)
    })
    
    # Ajouter le dataframe nettoyé
    dfs_clean.append(df_sans_outliers)

# Créer un DataFrame avec les résultats
df_outliers = pd.DataFrame(resultats)

# Trier par nombre d'outliers décroissant
df_outliers = df_outliers.sort_values('Nombre d\'outliers', ascending=False)

# Afficher le tableau complet
display(df_outliers)

# Créer le CSV nettoyé
df_clean = pd.concat(dfs_clean, ignore_index=True)
file_path_clean = f'../csv/STEP04/STEP04_maisons_dept{dept}.csv'
os.makedirs(os.path.dirname(file_path_clean), exist_ok=True)
df_clean.to_csv(file_path_clean, index=False, encoding='utf-8')
print(f"Fichier nettoyé sauvegardé : {file_path_clean}")

,Ville,Nombre de maisons,Nombre d'outliers,Moyenne avant outliers (€/m²),Moyenne après outliers (€/m²)
22,Carantec 29660,73,22,4938.75,4084.04
210,Quimper 29000,255,11,2518.86,2479.44
54,La Foret-Fouesnant 29940,67,10,3752.33,3655.96
142,Penmarch 29760,88,10,2918.98,2607.46
137,Morlaix 29600,70,8,3196.94,2101.11
18,Brest 29200,201,8,2661.49,2572.43
136,Moelan-Sur-Mer 29350,92,6,3710.80,3385.94
180,Plouhinec 29780,47,5,2585.98,2429.48
55,Fouesnant 29170,98,5,3612.98,3467.90
265,Tregunc 29910,64,5,3643.39,3337.54


Fichier nettoyé sauvegardé : ../csv/STEP04/STEP04_maisons_dept29.csv


### Vérification

In [12]:
import pandas as pd
import os

# Définir les départements à traiter
departements = ['22', '29', '35', '56']

# Stocker les résultats
resultats = []

for dept in departements:
    file_step02 = f'../csv/STEP02/STEP02_maisons_dept{dept}.csv'
    file_step04 = f'../csv/STEP04/STEP04_maisons_dept{dept}.csv'
    
    # Charger les DataFrames
    df02 = pd.read_csv(file_step02, encoding='utf-8')
    df04 = pd.read_csv(file_step04, encoding='utf-8')
    
    n02 = len(df02)
    n04 = len(df04)
    diff = n02 - n04
    
    resultats.append({
        'Département': dept,
        'STEP02': n02,
        'STEP04': n04,
        'Différence': diff
    })

# Afficher le tableau récapitulatif
df_resultats = pd.DataFrame(resultats)
display(df_resultats)

,Département,STEP02,STEP04,Différence
0,22,4225,3892,333
1,29,5335,4695,640
2,35,5867,5391,476
3,56,5898,5338,560
